# Dependencies
Install necessary dependencies:
- tokenizers, transformers for tokenization
- torch for training

In [1]:
!pip install tokenizers==0.21.1 transformers==4.51.3 torch==2.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 89.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [2]:
import torch
torch.cuda.device_count()

1

In [3]:
# Mount Google Drive so that we can access the training data
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
MINI_SOURCE_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh-mini.en"
MINI_TARGET_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh-mini.zh"
SOURCE_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh.en"
TARGET_FILE = "/content/drive/MyDrive/transformer-learning/en-zh.txt/TED2013.en-zh.zh"

# Decoder-only
Models like GPT are deocder-only, because they keep predicting the next token by what are already generated, we don't need an encoder in this process. In terms of implementation, we need to modify:
1. The decoder block to remove the encoder-decoder attention, since we have no encoder anymore.
2. The training loop should be updated.

We can implement the `DecoderOnlyBlock` with the cross attention removed, very simple.

In [5]:
from torch import nn

class DecoderOnlyBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        # The masked Multi-head attention
        self.masked_self_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        self.dropout1 = nn.Dropout(dropout)
        # The feed forward (FFN) part
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model) # Normalizes over the d_model dimension
        self.dropout2 = nn.Dropout(dropout) # Dropout after FFN residual

    def forward(
        self,
        target_input,
        trg_padding_mask=None,
        trg_attn_mask=None,
    ):
        """
        Decoder will need to take in the encoder output as part of the input
        """
        residual_input = target_input
        # --- Masked Multi-Head Self-Attention ---
        # Query, Key, Value are the same (from the decoder's path)
        # Pass the trg_attn_mask (causal), and trg_padding_mask
        masked_attn_output, _ = self.masked_self_attention(
            query=target_input,
            key=target_input,
            value=target_input,
            attn_mask=trg_attn_mask,
            key_padding_mask=trg_padding_mask,
        )

        # --- Add & Norm 1 ---
        # Add residual connection (input to this sub-layer)
        output_after_self_attn = residual_input + masked_attn_output
        # Apply dropout
        output_after_self_attn = self.dropout1(output_after_self_attn)
        # Apply Layer Norm
        norm1_output = self.norm1(output_after_self_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm1_output # Input to Feed-Forward Network

        # --- Feed-Forward Network ---
        # FFN operates independently on the last dimension
        ffn_output = self.linear_relu_stack(norm1_output) # Shape (batch_size, target_seq_len, d_model)
        # Apply dropout
        ffn_output = self.dropout2(ffn_output) # Dropout after FFN output

        # --- Add & Norm 3 ---
        # Add residual connection (input to this sub-layer)
        output_after_ffn = residual_input + ffn_output
        # Apply Layer Norm
        norm2_output = self.norm2(output_after_ffn) # Shape (batch_size, target_seq_len, d_model)

        return norm2_output # Output of the Decoder Block

Next, we will implement a decoder only transformer.

In [6]:
import math
import torch
from torch import nn

class DecoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        vocab_size_target: int,
        num_decoder_layers: int,
        max_seq_len: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_model = d_model
        # Embeddings
        self.target_embeddings = nn.Embedding(vocab_size_target, d_model)
        # Positional encoding is fixed, therefore we register it in buffer
        pe = self._generate_fixed_positional_encoding(max_seq_len, d_model)
        self.register_buffer('positional_encoding', pe)
        # Decoder layers
        self.decoder_stack = nn.ModuleList([
            DecoderOnlyBlock(d_model, heads, d_ff, dropout) for _ in range(num_decoder_layers)
        ])
        # Output layer
        self.output_layer = nn.Linear(d_model, vocab_size_target)
        # Initialize weights
        self._initialize_parameters()

    def _initialize_parameters(self):
        # Common initialization for Transformer weights
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _generate_fixed_positional_encoding(self, max_seq_len: int, d_model: int):
        """Generate a matrix for fixed positional encoding base on the max_seq_len"""
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Add batch dimension (1, max_seq_len, d_model)
        return pe # This will be added to embeddings

    def decode(
        self,
        trg_tokens,
        trg_padding_mask=None,
        trg_attn_mask=None,
    ):
      trg_embed = self.target_embeddings(trg_tokens) * math.sqrt(self.d_model) # Scale embeddings
      trg_embed = trg_embed + self.positional_encoding[:, :trg_tokens.size(1), :]
      trg_embed = nn.Dropout(self.dropout)(trg_embed)
      decoder_output = trg_embed
      for decoder_layer in self.decoder_stack:
        decoder_output = decoder_layer(
            decoder_output,
            trg_padding_mask=trg_padding_mask,
            trg_attn_mask=trg_attn_mask,
        )
      output_logits = self.output_layer(decoder_output)
      return output_logits

    def forward(
        self,
        trg_tokens,
        trg_padding_mask=None,
    ):
        """
        This is a high-level forward pass outline. Actual implementation needs mask handling.
        Masks need to be generated based on src_tokens and trg_tokens padding
        Lookahead mask for decoder self-attention also needs to be generated.
        """
        # Decoder Pass
        # Generate causal mask for decoder self-attention (shape seq_len, seq_len)
        causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(
            trg_tokens.size(1),
            device=trg_tokens.device
        )
        return self.decode(
            trg_tokens,
            trg_padding_mask=trg_padding_mask,
            trg_attn_mask=causal_mask,
        )

# Training
The way of training a decoder only transformer is slightly different because now we don't have an encoder. We will:
1. Prepare the training data like previously discussed
2. Train a tokenizer on our full English data
3. Format the training data into:
  1. Input sequences that starts with the `<SOS>` token (because remember, decoder input is shifted by one)
  2. Target sequences that does not start with the `<SOS>` token
4. Feed the input sequences into the model, get the prediections, and evaluate the loss against the target sequences.

First step, prepare the training data, we will again use the mini data for learning purpose.

In [7]:
from torch.utils.data import Dataset

class EnglishDataset(Dataset):
    def __init__(self, src_path: str):
        with open(src_path, 'r', encoding='utf-8') as f:
            self.src_lines = [line.strip() for line in f]

    def __len__(self):
        return len(self.src_lines)

    def __getitem__(self, idx):
        return self.src_lines[idx]

In [8]:
dataset = EnglishDataset(src_path=MINI_SOURCE_FILE)
dataset[0:5]

['http://www.ted.com/talks/stephen_palumbi_following_the_mercury_trail.html',
 "There's a tight and surprising link between the ocean's health and ours, says marine biologist Stephen Palumbi. He shows how toxins at the bottom of the ocean food chain find their way into our bodies, with a shocking story of toxic contamination from a Japanese fish market. His work points a way forward for saving the oceans' health -- and humanity's.",
 'fish,health,mission blue,oceans,science',
 '899',
 'Stephen Palumbi: Following the mercury trail']

In [9]:
from torch.utils.data import DataLoader, random_split

train_data, test_data = random_split(dataset, [0.8, 0.2])
print("Training data size:", len(train_data), ", Test data size:", len(test_data))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=True)

sample_train_input = next(iter(train_loader))
print(sample_train_input[0])

Training data size: 80 , Test data size: 20
And those simple themes aren't really themes about the complex science of what's going on, but things that we all pretty well know.


Then we will prepare the tokenizer, we will train it against the full dataset so that it can learn richer tokens.

In [10]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

special_tokens = ["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]", "[SOS]", "[EOS]"]

# Train one tokenizer for the English text
raw_en_tokenizer = Tokenizer(BPE())
raw_en_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=special_tokens)
raw_en_tokenizer.train(
  files=[SOURCE_FILE],
  trainer=trainer
)

In [11]:
from transformers import PreTrainedTokenizerFast

en_tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw_en_tokenizer)
en_tokenizer.add_special_tokens({'pad_token': '[PAD]', 'bos_token': '[SOS]', 'eos_token': '[EOS]'})

# Padding base on max_length, it will keep adding <PAD> until it reaches max_length
print(en_tokenizer("You smell money", return_tensors="pt", padding="max_length", max_length=16))

{'input_ids': tensor([[ 360, 4015,  936,    3,    3,    3,    3,    3,    3,    3,    3,    3,
            3,    3,    3,    3]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


Finally, prepare the training function.

In [12]:
def get_padding_mask(input_ids, pad_token_id, device):
      return torch.where(
          input_ids.to(device) == pad_token_id,
          torch.tensor(float('-inf'), device=device),
          torch.tensor(0.0, device=device),
      )

In [13]:
import torch

def train_one_epoch(transformer, train_loader, en_tokenizer, optimizer, criterion, device, max_seq_len, limit_batches=None):
    running_loss = 0.0
    total_batches = len(train_loader)
    processed_batches = 0

    pad_token_id = en_tokenizer.pad_token_id
    sos_token_id = en_tokenizer.bos_token_id

    if pad_token_id is None or sos_token_id is None:
         raise ValueError("Target tokenizer must have pad_token_id and bos_token_id defined.")

    transformer.train() # Set model to training mode
    for input_seq in train_loader:
        optimizer.zero_grad()

        # Tokenization
        # input_seq shape (batch_size, max_seq_len)
        en_tokens = en_tokenizer(
            input_seq, return_tensors="pt", padding="max_length", truncation=True, max_length=max_seq_len
        )
        en_input_ids = en_tokens["input_ids"].to(device)

        # Prepare padding mask
        decoder_padding_mask = get_padding_mask(
            en_input_ids, en_tokenizer.pad_token_id, device
        )

        # Prepare Decoder Input (shifted right: <SOS> + target tokens[:-1])
        batch_size = en_input_ids.size(0)
        decoder_input_ids = torch.full(
            (batch_size, 1), sos_token_id, device=device, dtype=en_input_ids.dtype
        )
        decoder_input_ids = torch.cat(
            [decoder_input_ids, en_input_ids[:, :-1]], dim=1
        )
        target_tokens_for_loss = en_input_ids

        # Forward pass
        # The main model's forward method is assumed to accept these additive masks
        # and handle causal mask generation internally, combining it with decoder_padding_mask for decoder self-attention
        # and passing encoder_padding_mask to decoder cross-attention.
        predictions = transformer(
            trg_tokens=decoder_input_ids, # Pass the shifted decoder input
            trg_padding_mask=decoder_padding_mask, # Additive mask for decoder self-attention (combined with causal) and cross-attention
        )

        # Calculate Loss
        # CrossEntropyLoss expects input (N, C) and target (N)
        # It will ignore the pad_token_id in target_tokens_for_loss
        loss = criterion(
            predictions.view(-1, predictions.size(-1)),
            target_tokens_for_loss.view(-1)
        )

        # Backpropagation and Optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        processed_batches += 1

        # Check if limit is reached
        if limit_batches is not None and processed_batches >= limit_batches:
            break

    # Calculate average loss based on processed batches
    avg_loss = running_loss / processed_batches if processed_batches > 0 else 0.0
    return avg_loss

Let's start training the model by running some epochs.

In [15]:
import torch
from torch import nn
import torch.optim as optim

num_epochs = 100
max_seq_len = 16
if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")
transformer_model = DecoderOnlyTransformer(
    d_model=64,
    heads=2,
    vocab_size_target=len(en_tokenizer),
    num_decoder_layers=2,
    max_seq_len=max_seq_len,
    d_ff=256,
    dropout=0.1
)
optimizer = optim.Adam(transformer_model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=en_tokenizer.pad_token_id) # Ignore padding in loss
transformer_model.to(device) # Move model to device

# Keep training until the loss is less than 2
loss_limit = 0.01
current_epoch = 0
train_loss = float('inf')
max_epoch = 1500
report_every = 50
while train_loss > loss_limit:
    # Train for one epoch
    train_loss = train_one_epoch(
        transformer_model,
        train_loader,
        en_tokenizer,
        optimizer,
        criterion,
        device,
        max_seq_len,
    )
    current_epoch += 1
    if current_epoch % report_every == 0:
        print(f"Epoch {current_epoch}/{max_epoch}, Loss: {train_loss:.4f}")

    if current_epoch > max_epoch:
        print(f"Did not reach {loss_limit} after {max_epoch} epochs. Loss: {train_loss:.4f}. Training stopped.")
        break

print(f"Final loss: {train_loss:.4f}")
print("Training finished.")

Epoch 50/1500, Loss: 4.8175
Epoch 100/1500, Loss: 2.1042
Epoch 150/1500, Loss: 0.8741
Epoch 200/1500, Loss: 0.5362
Epoch 250/1500, Loss: 0.4440
Epoch 300/1500, Loss: 0.4001
Epoch 350/1500, Loss: 0.3747
Epoch 400/1500, Loss: 0.3767
Epoch 450/1500, Loss: 0.3649
Epoch 500/1500, Loss: 0.3581
Epoch 550/1500, Loss: 0.3603
Epoch 600/1500, Loss: 0.3541
Epoch 650/1500, Loss: 0.3543
Epoch 700/1500, Loss: 0.3413
Epoch 750/1500, Loss: 0.3480
Epoch 800/1500, Loss: 0.3551
Epoch 850/1500, Loss: 0.3386
Epoch 900/1500, Loss: 0.3412
Epoch 950/1500, Loss: 0.3407
Epoch 1000/1500, Loss: 0.3394
Epoch 1050/1500, Loss: 0.3332
Epoch 1100/1500, Loss: 0.3340
Epoch 1150/1500, Loss: 0.3269
Epoch 1200/1500, Loss: 0.3380
Epoch 1250/1500, Loss: 0.3389
Epoch 1300/1500, Loss: 0.3289
Epoch 1350/1500, Loss: 0.3328
Epoch 1400/1500, Loss: 0.3329
Epoch 1450/1500, Loss: 0.3346
Epoch 1500/1500, Loss: 0.3362
Did not reach 0.01 after 1500 epochs. Loss: 0.3435. Training stopped.
Final loss: 0.3435
Training finished.


Let's make some predictions using the trained model.

In [36]:
def predict(transformer, en_tokenizer, device, max_seq_len, init_input = None):
    transformer.eval() # Set model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations

        sos_token_id = en_tokenizer.bos_token_id
        eos_token_id = en_tokenizer.eos_token_id
        pad_token_id = en_tokenizer.pad_token_id

        # Start by creating a target input with a <SOS> token only
        # Shape (1, 1)
        decoder_input_ids = torch.tensor([[sos_token_id]], device=device)

        # If init_input is provided, tokenize it and add a <SOS> token at front,
        # otherwise, simply start with a <SOS> token.
        if init_input is not None:
          init_input_tokens = en_tokenizer(init_input, return_tensors="pt")
          init_input_input_ids = init_input_tokens["input_ids"].to(device)
          decoder_input_ids = torch.cat(
              [torch.tensor([[sos_token_id]], device=device), init_input_input_ids],
              dim=1,
          )[:, :max_seq_len]
        else:
          decoder_input_ids = torch.tensor([[sos_token_id]], device=device)

        for _ in range(max_seq_len):
            current_seq_len = decoder_input_ids.size(1)
            # --- Prepare Decoder Input Tensor for Model ---
            # Pad current decoder input sequence to max_seq_len
            # Fill the padded_decoder_input_ids with <PAD>
            # Shape: (1, max_seq_len)
            padded_decoder_input_ids = torch.full(
                (decoder_input_ids.size(0), max_seq_len),
                pad_token_id,
                device=device,
                dtype=decoder_input_ids.dtype
            )
            # Place whatever we have to the padded_decoder_input_ids
            # Shape: (1 ,max_seq_len)
            padded_decoder_input_ids[:, :current_seq_len] = decoder_input_ids

            # Prepare padding mask
            # Shape: (1, max_seq_len)
            decoder_padding_mask = get_padding_mask(
                padded_decoder_input_ids,
                pad_token_id,
                device
            )

            # Prepare the attention mask
            # Shape: (max_seq_len, max_seq_len)
            # It is a triangular matrix where lower triangle is 0 and upper
            # triangle is float('-inf').
            causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(
                max_seq_len,
                device=padded_decoder_input_ids.device
            )

            # Get decoder output logits
            # Shape: (1, max_seq_len, vocab_size)
            decoder_output_logits = transformer.decode(
                padded_decoder_input_ids,
                trg_padding_mask=decoder_padding_mask,
                trg_attn_mask=causal_mask,
            )

            # Get the logit of the next token
            # The shape of the decoder output is (1, max_seq_len, vocab_size)
            # We will select the logit at the current_seq_len - 1 because this
            # is the token that we are predicting.
            # Shape: (1, vocab_size)
            next_token_logits = decoder_output_logits[:, current_seq_len - 1, :]
            # Get predicted token ID (Greedy decoding)
            # Apply softmax on the last dimension of next_token_logits, which has
            # size vocab_size.
            # Shape: (1, vocab_size)
            next_token_probs = torch.softmax(next_token_logits, dim=-1)

            # Use argmax to select the token id with the highest probability
            predicted_next_token_id = torch.argmax(next_token_probs, dim=-1).item()

            # Append the predicted token to the sequence for the next iteration
            # Shape (1, current_seq_len + 1)
            decoder_input_ids = torch.cat(
                [decoder_input_ids, torch.tensor([[predicted_next_token_id]], device=device)],
                dim=1
            )

            if predicted_next_token_id == eos_token_id:
                break # Stop if EOS token is predicted
            if current_seq_len + 1 >= max_seq_len:
                break # Stop if max length is reached

        # Convert batch into one single list
        # Shape (max_seq_len,)
        predicted_ids_list = decoder_input_ids[0].tolist()
        if eos_token_id in predicted_ids_list:
            eos_index = predicted_ids_list.index(eos_token_id)
        else:
            eos_index = len(predicted_ids_list)

        # Exclude SOS token (first token) and EOS token (if present)
        translation_ids = predicted_ids_list[1:eos_index]

        # Convert token IDs back to a string
        translation = en_tokenizer.decode(translation_ids, skip_special_tokens=True)

        return translation

In [37]:
predict(transformer_model, en_tokenizer, device, max_seq_len)

"And that ' s what we ' re trying to preserve when we say ,"

In [54]:
sample_train_input = next(iter(train_loader))[0]
print("Expected:", sample_train_input)
sample_train_input = " ".join(sample_train_input.split(" ")[:5])
print("Input:", sample_train_input)
predicted = predict(transformer_model, en_tokenizer, device, max_seq_len, init_input=sample_train_input)
print("Actual:", predicted)

Expected: Those females are passing the PCBs in the fat of their own mother's milk into their offspring, and their offspring don't survive.
Input: Those females are passing the
Actual: Those females are passing the PCBs in the fat of their own mother ' s


In [57]:
sample_test_input = next(iter(test_loader))[0]
print("Expected:", sample_test_input)
sample_test_input = " ".join(sample_test_input.split(" ")[:5])
print("Input:", sample_test_input)
predicted = predict(transformer_model, en_tokenizer, device, max_seq_len, init_input=sample_test_input)
print("Actual:", predicted)

Expected: And that's exactly what happens with PDBs in this food pyramid: They accumulate into the top of it.
Input: And that's exactly what happens
Actual: And that ' s exactly what happens , who was a lot of different ways


That's it, we have trained a decoder only, autoregressive transformer that generates based on our input. There is of course still some problems:
1. The model did not learn where to stop, because we did not put in the `<EOS>` token in our training set.
2. The model perform badly on the testing set because it is overfitted for the training set.